<a href="https://colab.research.google.com/github/OishiNikku/GPT-TS/blob/main/notebooks/pka_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Calculate microstate pKa values

Code and more documentation at:
https://github.com/mayrf/pkasolver

In [1]:
# @title Initializing Condacolab
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [2]:
# @title Check that everything is fine

import condacolab
condacolab.check()

✨🍰✨ Everything looks OK!


In [3]:
# @title Installing dependencies and pkasolver package (this might take up to 5 minutes)

print('📦 Installing dependencies ...')
!mamba install -c conda-forge rdkit > /dev/null
print('🔥 Installing PyTorch and PyG ...')
!pip install torch==1.13.1+cpu -f https://download.pytorch.org/whl/cpu/torch_stable.html > /dev/null
!pip install torch-scatter -f https://data.pyg.org/whl/torch-1.13.1+cpu.html > /dev/null
!pip install torch-sparse -f https://data.pyg.org/whl/torch-1.13.1+cpu.html > /dev/null
!pip install torch-spline-conv torch-geometric==2.0.1 -f https://data.pyg.org/whl/torch-1.13.1+cpu.html > /dev/null
!pip install cairosvg svgutils molvs > /dev/null
print('✔️ Installing pkasolver package ...')
!pip install -q git+https://github.com/mayrf/pkasolver.git > /dev/null
print("🎉 Done!")

📦 Installing dependencies ...
🔥 Installing PyTorch and PyG ...
✔️ Installing pkasolver package ...
🎉 Done!


In [15]:
# @title Predict pKa values
from pkasolver.query import QueryModel
from pkasolver.ml_architecture import GINPairV1
import pickle
import pkasolver
import torch
from os import path
from rdkit import Chem
from pkasolver.query import calculate_microstate_pka_values, draw_pka_reactions
from IPython.display import display

# load trained model
base_path = path.dirname(pkasolver.__file__)
# get input

buffers_smiles = pd.read_csv("/Users/wojtek/Desktop/New_LNP_Buffers/Organic_Acid_Metabolite_Library_of_Standards.csv", encoding='latin-1')

#buffers_smiles.at[1, 'Pka']=10
#print(buffers_smiles.head())

for i in range(0, len(buffers_smiles) - 1):
	smiles_string = buffers_smiles.at[i, "SMILES"]
	pka = pka_lookup_pubchem(smiles_string, "smiles")
	if pka == None:
		print("Passed: " + str(smiles_string))
		continue
	print('Example looking up by SMILES:')

	# # Look up pKa using pka_lookup_pubchem():
	# print(f'pKa from Pubchem using smiles:\n{pka_lookup_pubchem(smiles_string)}')
	print('pKa from Pubchem using smiles: ' + str(pka['pKa']))
	buffers_smiles.at[i, 'Pka'] = pka['pKa']
	buffers_smiles.at[i, 'Source'] = "PubChem"
buffers_smiles.to_csv('buffers_smiles.csv')


smiles = "CC(=O)O"  # @param {type:"string"}
# convert from Smiles to rdkit mol
mol = Chem.MolFromSmiles(smiles)
################################################
################################################
# calculate microstate pka values
protonation_states = calculate_microstate_pka_values(mol, only_dimorphite=False)
################################################
try:
  for i in range(len(protonation_states)):
    print(protonation_states[i].pka)
    print(protonation_states[i].pka_stddev)
except TypeError:
  print(protonation_states.pka)
  print(protonation_states.pka_stddev)
# draw the micostate pka values

[query.py:297 - calculate_microstate_pka_values()] Using dimorphite-dl to identify protonation sites.


Proposed mol at pH 7.4: CC(=O)[O-]
4.1944623470306395
0.2651004022650879


In [ ]:
# @title Report SMILE list with pKa values

print("😀################################😀")
for i in range(len(protonation_states)):
    state = protonation_states[i]
    print(
        Chem.MolToSmiles(state.protonated_mol),
        Chem.MolToSmiles(state.deprotonated_mol),
    )
    print(state.pka)
print("😀################################😀")
